# Train QTDB 1-channel diffusion denoising model on Kaggle

Notebook n?y train Ph??ng ?n A ?? ??i chi?u s?t b?i g?c:

- D? li?u clean: QT Database, m?i m?u l? m?t beat ECG 1 k?nh `(512, 1)` ? 360 Hz.
- D? li?u noise: MIT-BIH Noise Stress Test Database, ch? d?ng baseline wander `bw`.
- Train ri?ng checkpoint cho `noise_type=1` v? `noise_type=2` theo c?ch chia BW trong `eval_new.txt`.
- Model: `UNet1D` nghi?n c?u c?a b?n, nh?ng c?u h?nh 1-k?nh: `in_channels=2`, `out_channels=1`.
- Loss: multi-domain loss g?m L1 noise prediction + STFT magnitude loss.
- Output checkpoint l?u t?i `/kaggle/working/qtdb_1ch_checkpoints/`.

Tr??c khi ch?y: b?t GPU v? Internet trong Kaggle Notebook settings.


## 1. Setup


In [ ]:
from pathlib import Path
import gc
import math
import random
import time

import numpy as np
import pandas as pd
import scipy.signal as signal
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    import wfdb
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'])
    import wfdb

KAGGLE_WORKING = Path('/kaggle/working')
if not KAGGLE_WORKING.exists():
    raise RuntimeError('Notebook n?y ???c thi?t k? ?? ch?y tr?n Kaggle.')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))


## 2. Parameters


In [ ]:
TEST_RECORDS = [
    'sel123', 'sel233', 'sel302', 'sel307', 'sel820', 'sel853',
    'sel16420', 'sel16795', 'sele0106', 'sele0121', 'sel32',
    'sel49', 'sel14046', 'sel15814',
]

TARGET_FS = 360
BEAT_LENGTH = 512
CHANNELS = 1
NOISE_LEVEL_RANGE = (0.2, 1.99)

# Train c? 2 noise_type ?? eval ??ng protocol b?i g?c.
TRAIN_NOISE_TYPES = [1, 2]

# T?ng d?n sau khi ch?y th? th?nh c?ng.
MAX_TRAIN_RECORDS = None
MAX_TRAIN_BEATS = 8000
MAX_VAL_BEATS = 1000

EPOCHS = 30
BATCH_SIZE = 64
LR = 1e-4
NUM_DIFFUSION_STEPS = 50
BETA_START = 1e-4
BETA_END = 0.5
BETA_SCHEDULE = 'quad'
LAMBDA_TIME = 1.0
LAMBDA_FREQ = 0.1
VALID_EVERY = 1

BASE_FEATS = 80
EMB_DIM = 128
NUM_WORKERS = 2

OUTPUT_DIR = KAGGLE_WORKING / 'qtdb_1ch_checkpoints'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = KAGGLE_WORKING / 'qtdb_1ch_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)


## 3. Load QTDB clean beats


In [ ]:
BEAT_SYMBOLS = {
    'N', 'L', 'R', 'B', 'A', 'a', 'J', 'S', 'V', 'r', 'F', 'e', 'j', 'n', 'E', '/', 'f', 'Q', '?'
}


def get_qtdb_records():
    try:
        records = wfdb.get_record_list('qtdb')
    except Exception:
        records = wfdb.get_record_list('qtdb/1.0.0')
    return [r for r in records if not r.endswith('/')]


def read_qtdb_annotation(record_name):
    last_error = None
    for extension in ['atr', 'pu', 'q1c', 'q2c']:
        try:
            return wfdb.rdann(record_name, extension, pn_dir='qtdb/1.0.0'), extension
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f'No annotation for {record_name}: {last_error}')


def annotation_centers(annotation):
    symbols = np.asarray(annotation.symbol)
    samples = np.asarray(annotation.sample)
    beat_mask = np.asarray([symbol in BEAT_SYMBOLS for symbol in symbols])
    return samples[beat_mask] if beat_mask.any() else samples


def resample_to_target_fs(ecg, source_fs, target_fs=360):
    if int(source_fs) == int(target_fs):
        return ecg.astype(np.float32)
    target_len = int(round(ecg.shape[0] * target_fs / source_fs))
    return signal.resample(ecg, target_len, axis=0).astype(np.float32)


def select_eval_channel(ecg, channel_index=0):
    if ecg.ndim == 1:
        ecg = ecg[:, None]
    return ecg[:, channel_index:channel_index + 1].astype(np.float32)


def crop_beat(ecg, center, length=512):
    half = length // 2
    start = int(center) - half
    end = start + length
    if start < 0 or end > len(ecg):
        return None
    return ecg[start:end]


def normalize_segment(segment, eps=1e-8):
    segment = segment - np.median(segment, axis=0, keepdims=True)
    scale = np.max(np.abs(segment), axis=0, keepdims=True)
    return (segment / (scale + eps)).astype(np.float32)


def load_clean_beats(records, cache_name, max_records=None, max_beats=None):
    cache_path = CACHE_DIR / cache_name
    if cache_path.exists():
        cached = np.load(cache_path, allow_pickle=True)
        beats = cached['beats']
        meta = pd.DataFrame(cached['meta'].tolist())
        if max_beats is not None:
            beats = beats[:max_beats]
            meta = meta.iloc[:max_beats].reset_index(drop=True)
        print(f'Loaded cache {cache_path}:', beats.shape)
        return beats.astype(np.float32), meta

    selected_records = list(records)
    if max_records is not None:
        selected_records = selected_records[:max_records]

    beats = []
    meta = []
    for record_name in tqdm(selected_records, desc=cache_name):
        try:
            record = wfdb.rdrecord(record_name, pn_dir='qtdb/1.0.0')
            ann, ann_ext = read_qtdb_annotation(record_name)
            ecg = resample_to_target_fs(record.p_signal, record.fs, TARGET_FS)
            ecg = select_eval_channel(ecg, channel_index=0)
            ratio = TARGET_FS / float(record.fs)
            centers = np.round(annotation_centers(ann) * ratio).astype(int)
            kept = 0
            for center in centers:
                beat = crop_beat(ecg, center, BEAT_LENGTH)
                if beat is None:
                    continue
                beat = normalize_segment(beat)
                if np.all(np.isfinite(beat)) and beat.shape == (BEAT_LENGTH, CHANNELS):
                    beats.append(beat)
                    meta.append({'record': record_name, 'annotation': ann_ext, 'center_sample': int(center)})
                    kept += 1
                if max_beats is not None and len(beats) >= max_beats:
                    arr = np.stack(beats).astype(np.float32)
                    np.savez_compressed(cache_path, beats=arr, meta=np.array(meta, dtype=object))
                    return arr, pd.DataFrame(meta)
            print(f'{record_name}: kept {kept} beats')
            del record, ann, ecg, centers
            gc.collect()
        except Exception as exc:
            print(f'Skip {record_name}: {exc}')
            gc.collect()

    if not beats:
        raise RuntimeError('No beats loaded from QTDB.')
    arr = np.stack(beats).astype(np.float32)
    np.savez_compressed(cache_path, beats=arr, meta=np.array(meta, dtype=object))
    return arr, pd.DataFrame(meta)


all_records = get_qtdb_records()
test_set = set(TEST_RECORDS)
train_records = [r for r in all_records if r not in test_set]
if MAX_TRAIN_RECORDS is not None:
    train_records = train_records[:MAX_TRAIN_RECORDS]

train_clean_all, train_meta = load_clean_beats(train_records, 'train_clean_beats.npz', max_beats=MAX_TRAIN_BEATS)
print('Train clean:', train_clean_all.shape)
display(train_meta.groupby('record').size().rename('beats').reset_index().head())


## 4. Train/validation split and BW noise


In [ ]:
def load_bw_noise():
    record = wfdb.rdrecord('bw', pn_dir='nstdb/1.0.0')
    noise = resample_to_target_fs(record.p_signal, record.fs, TARGET_FS)
    if noise.shape[1] < 2:
        noise = np.tile(noise, (1, 2))
    return noise.astype(np.float32)


def split_noise_for_type(noise, noise_type):
    middle = len(noise) // 2
    if noise_type == 1:
        return noise[:middle, 0], noise[middle:, 1]
    if noise_type == 2:
        return noise[:middle, 1], noise[middle:, 0]
    raise ValueError('noise_type must be 1 or 2')


rng = np.random.default_rng(SEED)
indices = rng.permutation(len(train_clean_all))
val_size = min(MAX_VAL_BEATS, max(1, int(0.15 * len(indices))))
val_idx = indices[:val_size]
train_idx = indices[val_size:]
train_clean = train_clean_all[train_idx]
val_clean = train_clean_all[val_idx]

bw_noise = load_bw_noise()
noise_by_type = {nt: split_noise_for_type(bw_noise, nt)[0] for nt in TRAIN_NOISE_TYPES}
print('Train split:', train_clean.shape, 'Val split:', val_clean.shape)
for nt, noise in noise_by_type.items():
    print(f'noise_type={nt}: train BW samples={len(noise)}')


## 5. Dataset


In [ ]:
def amplitude_range(x, eps=1e-8):
    return float(np.max(x) - np.min(x) + eps)


def make_noisy_one(clean, noise, rng):
    start = int(rng.integers(0, len(noise) - BEAT_LENGTH))
    noise_patch = noise[start:start + BEAT_LENGTH].astype(np.float32)
    noise_patch = noise_patch - np.median(noise_patch)
    noise_patch = noise_patch[:, None]
    ase = amplitude_range(noise_patch) / amplitude_range(clean)
    level = float(rng.uniform(NOISE_LEVEL_RANGE[0], NOISE_LEVEL_RANGE[1]))
    alpha = level / ase
    return (clean + alpha * noise_patch).astype(np.float32), level


class QTDBNoisyDataset(Dataset):
    def __init__(self, clean_beats, noise_signal, seed=42):
        self.clean_beats = clean_beats.astype(np.float32)
        self.noise_signal = noise_signal.astype(np.float32)
        self.seed = int(seed)
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.clean_beats)

    def __getitem__(self, idx):
        rng = np.random.default_rng(self.seed + self.epoch * len(self.clean_beats) + int(idx))
        clean = self.clean_beats[idx]
        noisy, _ = make_noisy_one(clean, self.noise_signal, rng)
        clean_t = torch.from_numpy(clean).float().permute(1, 0)  # (1, 512)
        noisy_t = torch.from_numpy(noisy).float().permute(1, 0)
        return clean_t, noisy_t


## 6. Model and diffusion loss


In [ ]:
class HNFBlockUNet(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_sizes=(3, 5, 9, 15)):
        super().__init__()
        self.multi_convs = nn.ModuleList([
            nn.Conv1d(in_channels, out_channels // len(kernel_sizes), k, padding=k // 2, padding_mode='reflect')
            for k in kernel_sizes
        ])
        self.agg_conv = nn.Conv1d(out_channels, out_channels, 1)
        self.half_inst_norm = nn.InstanceNorm1d(out_channels // 2)
        self.act = nn.ReLU(inplace=True)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        out = torch.cat([conv(x) for conv in self.multi_convs], dim=1)
        out = self.agg_conv(out)
        half = out.shape[1] // 2
        out = torch.cat([self.half_inst_norm(out[:, :half, :]), out[:, half:, :]], dim=1)
        out = self.act(out)
        return out + self.residual(x)


class BridgeBlockUNet(nn.Module):
    def __init__(self, features, emb_dim=128):
        super().__init__()
        self.emb_dim = emb_dim
        self.film = nn.Sequential(nn.Linear(emb_dim, features * 2), nn.SiLU())

    def sinusoidal_embedding(self, x):
        x = x.view(-1)
        device = x.device
        half_dim = self.emb_dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x.unsqueeze(-1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)

    def forward(self, x, alpha_bar):
        emb = self.sinusoidal_embedding(alpha_bar)
        scale, shift = self.film(emb).chunk(2, dim=1)
        return x * (1 + scale.unsqueeze(-1)) + shift.unsqueeze(-1)


class SelfAttention1D(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        assert self.head_dim * num_heads == channels
        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, length = x.shape
        qkv = self.qkv(x).reshape(batch, 3, self.num_heads, self.head_dim, length)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        attn = torch.matmul(q.transpose(-2, -1), k) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = torch.matmul(attn, v.transpose(-2, -1)).transpose(-2, -1)
        return self.proj(out.reshape(batch, channels, length))


class UNet1D(nn.Module):
    def __init__(self, in_channels=2, base_channels=80, emb_dim=128, out_channels=1):
        super().__init__()
        self.enc1 = HNFBlockUNet(in_channels, base_channels)
        self.bridge1 = BridgeBlockUNet(base_channels, emb_dim)
        self.down1 = nn.Conv1d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.enc2 = HNFBlockUNet(base_channels * 2, base_channels * 2)
        self.bridge2 = BridgeBlockUNet(base_channels * 2, emb_dim)
        self.down2 = nn.Conv1d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.enc3 = HNFBlockUNet(base_channels * 4, base_channels * 4)
        self.bridge3 = BridgeBlockUNet(base_channels * 4, emb_dim)
        self.down3 = nn.Conv1d(base_channels * 4, base_channels * 8, kernel_size=4, stride=2, padding=1)
        self.enc4 = HNFBlockUNet(base_channels * 8, base_channels * 8)
        self.bridge4 = BridgeBlockUNet(base_channels * 8, emb_dim)
        self.attn = SelfAttention1D(base_channels * 8)
        self.up4 = nn.ConvTranspose1d(base_channels * 8, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.dec4 = HNFBlockUNet(base_channels * 8, base_channels * 4)
        self.up3 = nn.ConvTranspose1d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.dec3 = HNFBlockUNet(base_channels * 4, base_channels * 2)
        self.up2 = nn.ConvTranspose1d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)
        self.dec2 = HNFBlockUNet(base_channels * 2, base_channels)
        self.final = nn.Conv1d(base_channels, out_channels, kernel_size=1)

    def forward(self, x, cond, noise_scale):
        inp = torch.cat([x, cond], dim=1)
        e1 = self.bridge1(self.enc1(inp), noise_scale)
        e2 = self.bridge2(self.enc2(self.down1(e1)), noise_scale)
        e3 = self.bridge3(self.enc3(self.down2(e2)), noise_scale)
        e4 = self.bridge4(self.enc4(self.down3(e3)), noise_scale)
        e4 = self.attn(e4)
        d4 = self.dec4(torch.cat([self.up4(e4), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        return self.final(d2)


In [ ]:
def stft_loss(pred, target, n_fft=128, hop_length=64):
    batch, channels, length = pred.shape
    pred = pred.reshape(batch * channels, length)
    target = target.reshape(batch * channels, length)
    window = torch.hann_window(n_fft, device=pred.device)
    spec_pred = torch.stft(pred, n_fft=n_fft, hop_length=hop_length, window=window, return_complex=True)
    spec_target = torch.stft(target, n_fft=n_fft, hop_length=hop_length, window=window, return_complex=True)
    return F.mse_loss(torch.abs(spec_pred), torch.abs(spec_target))


def make_beta_schedule(schedule, n_timesteps, start, end):
    if schedule == 'linear':
        return torch.linspace(start, end, n_timesteps)
    if schedule == 'quad':
        return torch.linspace(start ** 0.5, end ** 0.5, n_timesteps) ** 2
    if schedule == 'sigmoid':
        values = torch.linspace(-6, 6, n_timesteps)
        return torch.sigmoid(values) * (end - start) + start
    raise ValueError(schedule)


def extract(buffer, t, shape):
    return buffer.gather(0, t).reshape((shape[0],) + (1,) * (len(shape) - 1))


class DDPM(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.model = base_model
        self.num_steps = NUM_DIFFUSION_STEPS
        betas = make_beta_schedule(BETA_SCHEDULE, NUM_DIFFUSION_STEPS, BETA_START, BETA_END)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = torch.cat([torch.ones(1), alphas_cumprod[:-1]])
        sqrt_alphas_cumprod_prev = torch.sqrt(torch.cat([torch.ones(1), alphas_cumprod]))

        posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

        self.sqrt_alphas_cumprod_prev_np = sqrt_alphas_cumprod_prev.numpy()
        self.register_buffer('betas', betas.float())
        self.register_buffer('alphas_cumprod', alphas_cumprod.float())
        self.register_buffer('alphas_cumprod_prev', alphas_cumprod_prev.float())
        self.register_buffer('sqrt_alphas_cumprod', torch.sqrt(alphas_cumprod).float())
        self.register_buffer('sqrt_one_minus_alphas_cumprod', torch.sqrt(1.0 - alphas_cumprod).float())
        self.register_buffer('log_one_minus_alphas_cumprod', torch.log(1.0 - alphas_cumprod).float())
        self.register_buffer('sqrt_recip_alphas_cumprod', torch.sqrt(1.0 / alphas_cumprod).float())
        self.register_buffer('sqrt_recipm1_alphas_cumprod', torch.sqrt(1.0 / alphas_cumprod - 1).float())
        self.register_buffer('posterior_variance', posterior_variance.float())
        self.register_buffer('posterior_log_variance_clipped', torch.log(torch.clamp(posterior_variance, min=1e-20)).float())
        self.register_buffer('posterior_mean_coef1', (betas * torch.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod)).float())
        self.register_buffer('posterior_mean_coef2', ((1.0 - alphas_cumprod_prev) * torch.sqrt(alphas) / (1.0 - alphas_cumprod)).float())

    def q_sample(self, x_start, continuous_sqrt_alpha_cumprod, noise):
        return continuous_sqrt_alpha_cumprod * x_start + (1 - continuous_sqrt_alpha_cumprod ** 2).sqrt() * noise

    def predict_start_from_noise(self, x_t, t, noise):
        return extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t - extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * noise

    def forward(self, clean, noisy_condition):
        batch = clean.shape[0]
        t = torch.randint(0, self.num_steps, (batch,), device=clean.device).long()
        sqrt_prev = torch.tensor(self.sqrt_alphas_cumprod_prev_np, dtype=torch.float32, device=clean.device)
        lower = sqrt_prev[t]
        upper = sqrt_prev[t + 1]
        continuous = lower + torch.rand(batch, device=clean.device) * (upper - lower)
        continuous = continuous.view(batch, 1, 1)
        noise = torch.randn_like(clean)
        x_noisy = self.q_sample(clean, continuous, noise)
        predicted_noise = self.model(x_noisy, noisy_condition, continuous.view(batch, 1))
        loss_time = F.l1_loss(predicted_noise, noise)
        x0_pred = self.predict_start_from_noise(x_noisy, t, predicted_noise)
        loss_freq = stft_loss(x0_pred, clean)
        return LAMBDA_TIME * loss_time + LAMBDA_FREQ * loss_freq

    @torch.no_grad()
    def ddim_sample_loop(self, condition, ddim_timesteps=15, ddim_eta=0.0, num_shots=1):
        device = condition.device
        total_steps = self.num_steps
        sample_steps = min(ddim_timesteps, total_steps)
        tau = [int(np.floor((total_steps / (sample_steps ** 2)) * (i ** 2))) for i in range(sample_steps + 1)]
        alphas_with_one = torch.cat([torch.ones(1, device=device), self.alphas_cumprod])
        out_accum = torch.zeros_like(condition)
        for _ in range(num_shots):
            x = torch.randn_like(condition)
            for i in reversed(range(1, sample_steps + 1)):
                t_value = tau[i]
                t_prev = tau[i - 1]
                noise_level = torch.full((x.shape[0], 1), float(torch.sqrt(alphas_with_one[t_value])), device=device)
                eps = self.model(x, condition, noise_level)
                a_t = alphas_with_one[t_value]
                a_prev = alphas_with_one[t_prev]
                x0_pred = (x - torch.sqrt(1.0 - a_t) * eps) / torch.sqrt(a_t)
                sigma_t = 0.0 if t_prev == 0 else ddim_eta * torch.sqrt((1.0 - a_prev) / (1.0 - a_t)) * torch.sqrt(torch.clamp(1.0 - a_t / a_prev, min=0.0))
                direction = torch.sqrt(torch.clamp(1.0 - a_prev - sigma_t ** 2, min=0.0)) * eps
                noise = sigma_t * torch.randn_like(x) if (ddim_eta > 0 and t_prev > 0) else 0.0
                x = torch.sqrt(a_prev) * x0_pred + direction + noise
            out_accum += x
        return out_accum / num_shots


## 7. Training loop


In [ ]:
def cosine_torch(a, b):
    return F.cosine_similarity(a.reshape(a.shape[0], -1), b.reshape(b.shape[0], -1), dim=1).mean().item()


@torch.no_grad()
def validate(model, val_loader):
    model.eval()
    losses = []
    cosines = []
    for clean, noisy in tqdm(val_loader, desc='valid', leave=False):
        clean = clean.to(DEVICE)
        noisy = noisy.to(DEVICE)
        loss = model(clean, noisy)
        denoised = model.ddim_sample_loop(noisy, ddim_timesteps=15, ddim_eta=0.0, num_shots=1)
        losses.append(loss.item())
        cosines.append(cosine_torch(clean, denoised))
    return float(np.mean(losses)), float(np.mean(cosines))


def train_one_noise_type(noise_type):
    print()
    print(f'=== Train noise_type={noise_type} ===')
    train_dataset = QTDBNoisyDataset(train_clean, noise_by_type[noise_type], seed=SEED + noise_type * 1000)
    val_dataset = QTDBNoisyDataset(val_clean, noise_by_type[noise_type], seed=SEED + noise_type * 2000)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    base_model = UNet1D(in_channels=2, base_channels=BASE_FEATS, emb_dim=EMB_DIM, out_channels=1).to(DEVICE)
    model = DDPM(base_model).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    best_val_loss = float('inf')
    history = []

    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()
        train_dataset.set_epoch(epoch)
        model.train()
        train_losses = []
        for clean, noisy in tqdm(train_loader, desc=f'epoch {epoch}/{EPOCHS}', leave=False):
            clean = clean.to(DEVICE, non_blocking=True)
            noisy = noisy.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = model(clean, noisy)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        if epoch % VALID_EVERY == 0:
            val_dataset.set_epoch(epoch)
            val_loss, val_cos = validate(model, val_loader)
        else:
            val_loss, val_cos = np.nan, np.nan

        row = {
            'noise_type': noise_type,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_cosine_ddim15': val_cos,
            'seconds': round(time.time() - start_time, 2),
        }
        history.append(row)
        print(row)

        if not np.isnan(val_loss) and val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = OUTPUT_DIR / f'qtdb_1ch_noise_type_{noise_type}_best.pth'
            torch.save(model.state_dict(), best_path)
            print('Saved best:', best_path)

        last_path = OUTPUT_DIR / f'qtdb_1ch_noise_type_{noise_type}_last.pth'
        torch.save(model.state_dict(), last_path)
        pd.DataFrame(history).to_csv(OUTPUT_DIR / f'qtdb_1ch_noise_type_{noise_type}_training_log.csv', index=False)
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()

    return pd.DataFrame(history)


## 8. Run training


In [ ]:
all_history = []
for noise_type in TRAIN_NOISE_TYPES:
    history = train_one_noise_type(noise_type)
    all_history.append(history)

history_df = pd.concat(all_history, ignore_index=True)
history_path = OUTPUT_DIR / 'qtdb_1ch_training_log_all.csv'
history_df.to_csv(history_path, index=False)
display(history_df.tail())
print('Saved all training outputs to:', OUTPUT_DIR)


## 9. Quick checkpoint sanity check


In [ ]:
# Ch?y th? m?t batch validation b?ng checkpoint best ?? ki?m tra output kh?ng b? l?i shape.
for noise_type in TRAIN_NOISE_TYPES:
    path = OUTPUT_DIR / f'qtdb_1ch_noise_type_{noise_type}_best.pth'
    if not path.exists():
        path = OUTPUT_DIR / f'qtdb_1ch_noise_type_{noise_type}_last.pth'
    model = DDPM(UNet1D(in_channels=2, base_channels=BASE_FEATS, emb_dim=EMB_DIM, out_channels=1)).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE), strict=True)
    model.eval()

    dataset = QTDBNoisyDataset(val_clean[:BATCH_SIZE], noise_by_type[noise_type], seed=999 + noise_type)
    loader = DataLoader(dataset, batch_size=min(BATCH_SIZE, len(dataset)), shuffle=False)
    clean, noisy = next(iter(loader))
    clean = clean.to(DEVICE)
    noisy = noisy.to(DEVICE)
    with torch.no_grad():
        denoised = model.ddim_sample_loop(noisy, ddim_timesteps=15, ddim_eta=0.0, num_shots=1)
    print(noise_type, path.name, 'clean', tuple(clean.shape), 'noisy', tuple(noisy.shape), 'denoised', tuple(denoised.shape), 'cosine', cosine_torch(clean, denoised))


## 10. Download/use outputs

Sau khi Kaggle ch?y xong, t?i c?c file sau t? `/kaggle/working/qtdb_1ch_checkpoints/` r?i upload l?n Google Drive ?? notebook Colab eval d?ng:

- `qtdb_1ch_noise_type_1_best.pth`
- `qtdb_1ch_noise_type_2_best.pth`
- c?c file `*_training_log.csv` ?? ??a v?o ph? l?c n?u c?n

Trong notebook Colab eval, tr?:

```python
CHECKPOINT_PATHS = {
    1: '/content/drive/MyDrive/phase1/checkpoints/qtdb_1ch_noise_type_1_best.pth',
    2: '/content/drive/MyDrive/phase1/checkpoints/qtdb_1ch_noise_type_2_best.pth',
}
```
